# Notebook 09: Market-Event Studies, Macro Stress Testing & Model Explainability
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 9 (Post-Modeling Empirical Evaluation & Risk Intelligence)  
**Repository:** `Indian_Equities_Liquidity_Risk_ML-main`  
**Status:** Fully Executable & Reproducible Research Notebook  

---

### Notebook Objectives:
1. **Stage 9.2 — Historical Event Studies:** Symmetrical `[-5, +5]` trading-session window analysis across 86 curated Indian market events (2018–2026), evaluating liquidity dislocations and machine learning early-warning risk signals.
2. **Stage 9.3 — Macro Stress Testing:** Cross-sectional analysis across Large, Mid, and Small caps during major systemic crisis episodes (COVID-19 lower circuit halts, IL&FS default, Ukraine invasion, Lok Sabha 2024 election whiplash, SEBI froth warnings).
3. **Stage 9.4 — Model Explainability:** Native scikit-learn permutation feature importance (PR-AUC loss) on out-of-sample data for champion `CFG_05_HGBT` and Mean Decrease in Impurity (MDI) for `CFG_04_RF_CANDIDATE_B`.
4. **Synthesis & Operational Takeaways:** Translation of quantitative microstructure findings into concrete execution and risk-management protocols for institutional trading in Indian equities.

In [ ]:
# Environment Setup and Library Imports
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10

# Define authoritative paths relative to repository root
REPO_ROOT = Path('..').resolve() if Path('..').resolve().name == 'Indian_Equities_Liquidity_Risk_ML-main' else Path('.').resolve()
DATA_DIR = REPO_ROOT / 'data' / 'interim' / 'phase9'
REPORTS_DIR = REPO_ROOT / 'reports' / 'phase9'
FIGURES_DIR = REPORTS_DIR / 'figures'

print(f'Repository Root: {REPO_ROOT}')
print(f'Phase 9 Data Directory: {DATA_DIR}')

## 1. Historical Event Studies (Stage 9.2)

In Stage 9.2, we aligned the 86 curated events from `data/events/market_events.csv` with our 2,176-trading-day history, constructing symmetrical `[-5, +5]` trading-session windows across 300 Indian equities (242,229 stock-event observations).

In [ ]:
# Load Stage 9.2 Event Impact Summaries
impact_summary_path = DATA_DIR / 'event_study_metrics' / 'event_impact_summary.csv'
impact_df = pd.read_csv(impact_summary_path)

print(f'Total Curated Events Analyzed: {len(impact_df)}')
print(f'Categories: {impact_df["event_category"].value_counts().to_dict()}')
print(f'Prediction Regimes: {impact_df["prediction_regime"].value_counts().to_dict()}')

# Display Top 10 Events by Corwin-Schultz Spread Shock
top_spread_shocks = impact_df.sort_values(by='spread_shock_pct', ascending=False).head(10)
top_spread_shocks[['event_id', 'event_name', 'event_category', 'prediction_regime', 'pre_spread_median', 'event_spread_median', 'spread_shock_pct']]

In [ ]:
# Category-Level Median Spread Shocks and Amihud Surges
cat_summary = impact_df.groupby('event_category').agg({
    'event_id': 'count',
    'spread_shock_pct': 'median',
    'amihud_shock_pct': 'median',
    'hgbt_prob_delta': 'median',
}).rename(columns={
    'event_id': 'Event Count',
    'spread_shock_pct': 'Median Spread Shock (%)',
    'amihud_shock_pct': 'Median Amihud Shock (%)',
    'hgbt_prob_delta': 'Median HGBT Prob Delta',
})
cat_summary.sort_values(by='Median Spread Shock (%)', ascending=False)

### Event Window Trajectory Visualization
Below we display the cross-sectional event window trajectory (`reports/phase9/figures/event_window_analysis/event_window_spread_and_amihud.png`), tracking spread widening and Amihud price impact across relative trading sessions $\tau \in [-5, +5]$.

In [ ]:
# Display Stage 9.2 Generated Figures
from IPython.display import Image, display

fig_path1 = FIGURES_DIR / 'event_window_analysis' / 'event_window_spread_and_amihud.png'
if fig_path1.exists():
    display(Image(filename=str(fig_path1), width=900))
else:
    print(f'Figure not found at: {fig_path1}')

## 2. Macro Stress Testing & Market-Cap Divergence (Stage 9.3)

In Stage 9.3, we selected seven representative historical stress episodes across distinct macro categories and regimes to quantify how liquidity evaporates across **Large-Cap, Mid-Cap, and Small-Cap** tiers.

In [ ]:
# Load Stage 9.3 Stress Episode Divergence and Clustering Artifacts
div_path = DATA_DIR / 'macro_stress_metrics' / 'stress_segment_divergence.csv'
div_df = pd.read_csv(div_path)

print('=== Small-Cap vs Large-Cap Liquidity Divergence across Stress Episodes ===')
small_large_div = div_df[div_df['segment'] == 'DIVERGENCE_SMALL_VS_LARGE']
small_large_div[['episode_id', 'episode_name', 'category', 'regime', 'spread_shock_abs', 'amihud_shock_pct', 'if_flag_shock_abs']]

In [ ]:
# Load Non-Parametric Significance Tests
test_path = DATA_DIR / 'macro_stress_metrics' / 'stress_statistical_tests.csv'
stat_df = pd.read_csv(test_path)

print(f'Total Non-Parametric Hypothesis Tests Conducted: {len(stat_df)}')
significant_tests = stat_df[stat_df['is_significant_005'] == True]
print(f'Significant Tests at alpha = 0.05: {len(significant_tests)} / {len(stat_df)}')
significant_tests[['episode_id', 'comparison', 'test_type', 'sample_size', 'statistic', 'p_value']]

In [ ]:
# Display Stage 9.3 Visualizations: Spread Widening and Isolation Forest Clustering
fig_stress1 = FIGURES_DIR / 'liquidity_stress' / 'stress_period_spread_widening.png'
fig_stress2 = FIGURES_DIR / 'liquidity_stress' / 'stress_segment_divergence.png'
fig_stress3 = FIGURES_DIR / 'liquidity_stress' / 'isolation_forest_stress_clustering.png'

for fpath in [fig_stress1, fig_stress2, fig_stress3]:
    if fpath.exists():
        display(Image(filename=str(fpath), width=850))

## 3. Model Explainability & Feature Importance (Stage 9.4)

In Stage 9.4, we computed Permutation Feature Importance for the champion **`CFG_05_HGBT`** model using PR-AUC (`average_precision`) degradation across 5 repeats, and compared it against the native tree-split impurity reduction (MDI) of **`CFG_04_RF_CANDIDATE_B`**.

In [ ]:
# Load Top 10 Risk Drivers and Cross-Model Comparison
top10_path = DATA_DIR / 'explainability' / 'top_10_risk_drivers.csv'
top10_df = pd.read_csv(top10_path)

print('=== CFG_05_HGBT Champion: Top 10 Predictive Risk Drivers ===')
top10_df[['rank', 'feature', 'family', 'importance_mean_prauc_loss', 'importance_std', 'financial_interpretation']]

In [ ]:
# Display Feature Importance Figures
fig_exp1 = FIGURES_DIR / 'feature_importance' / 'hgbt_permutation_importance.png'
fig_exp2 = FIGURES_DIR / 'feature_importance' / 'importance_comparison_hgbt_vs_rf.png'

for fpath in [fig_exp1, fig_exp2]:
    if fpath.exists():
        display(Image(filename=str(fpath), width=850))

## 4. Key Conclusions & Operational Recommendations

### Core Discoveries:
1. **Amihud Illiquidity is King:** `amihud_ma20` is the single most dominant predictor of High Risk Events in Indian equities, causing a 22.3% drop in PR-AUC when permuted.
2. **Asymmetric Downside Sensitivity:** Models rely far more heavily on downside semivariance (`downside_vol_20d`, rank #3) than total volatility (`vol_20d`, rank #9), reflecting that panic liquidation dries up order book liquidity.
3. **Small-Cap Vulnerability & Asymmetry:** Small and mid caps experience severe liquidity evaporation during systemic shocks, with Isolation Forest anomaly flag rates surging up to 5.7x during macro dislocations.
4. **Out-of-Sample Temporal Stability:** Ordinal feature rankings remain remarkably invariant between the Walk-Forward OOS dataset (2023–2025) and the Final Holdout dataset (2025–2026), verifying that our model has learned genuine microstructure dynamics rather than transitory noise.

---
**End of Notebook 09.** All underlying interim datasets and reports are frozen and archived under `data/interim/phase9/` and `reports/phase9/`.